# Cylindrical refolded isocline — LDI and truth meshes

The horizon surfaces of the LDI fit for the 3-D figures, from the model
`07b_cylindrical_isocline_ldi` saved (`results/restore.pt`), and the true surfaces. Nothing is
fitted here. (The Surfe meshes are written by `07c`, which has the Surfe model.)

- **LDI** (`results/meshes_ldi.npz`): the level surface of φ at each horizon's isovalue, by
  octree dual contouring (`GeoEvent.mesh`), over the box from 07a. If the fit has interior
  extrema, the small closed level surface just inside each is meshed too, as for Surfe.
- **Truth** (`results/meshes_truth.npz`): each horizon's truth polylines from 07a swept along
  the hinge across the box. The same arrays as the fitted meshes, so the figures treat all
  of them alike.

## Setup

In [4]:
from pathlib import Path

import numpy as np
import torch

import curlew
from curlew.io import loadModel

import sys
sys.path.insert(0, str(Path("..").resolve()))  # publication/: figstyle, diagnostics, exploded3d
import diagnostics
import isocline_inputs
import flat_reference   # the restore's reference, needed to load the saved model

curlew.device = "cpu"
curlew.dtype = torch.float32
curlew.default_dim = 3

RES_DIR = isocline_inputs.RESULTS_DIR
data = isocline_inputs.load_processed()
box = data["box"]
ldi = np.load(RES_DIR / "ldi.npz")
model = loadModel(RES_DIR / "restore.pt")
event = model.events[0]
crit = diagnostics.unpack_critical(ldi)
print("LDI critical points:", crit.counts)

LDI critical points: {'maximum': 0, 'minimum': 0, 'saddle': 0, 'degenerate': 0}


## LDI surfaces

`MESH_DEPTH` is the octree depth (2^depth cells per axis); `QEF_TOL` places each
dual-contouring vertex (1 = the mass point, smooth sheets; smaller keeps sharp features).
For each interior extremum, the level surface `SHELL_DELTA` of the level spacing inside it is
meshed within `SHELL_HALF` m of it, as in 07c.

In [5]:
MESH_DEPTH = 7
QEF_TOL = 0.5
SHELL_DELTA = 0.08         # shell level, as a fraction of the level spacing
SHELL_HALF = 40.0          # m

isovalues = ldi["isovalues"]
spacing = float(np.min(np.abs(np.diff(isovalues))))


def mesher(level, bounds):
    v, f = event.mesh(iso=float(level), bounds=np.asarray(bounds), max_depth=MESH_DEPTH, qef_tol=QEF_TOL)
    return np.asarray(v), np.asarray(f)


def value_at(p):
    _, v = event.field.gradient(torch.as_tensor(p[None], dtype=curlew.dtype), normalize=False, return_value=True)
    return float(v.detach().reshape(-1)[0])


MESH_PAD_X = 60.0                              # m added to the box on the -x side, so the renders can show
mesh_box = box + np.array([[-MESH_PAD_X, 0.0, 0.0], [0.0, 0.0, 0.0]])   # the closure of the contacts there

extrema = [(p, k, value_at(p)) for p, k in zip(crit.points, crit.kinds) if k in ("maximum", "minimum")]
diagnostics.export_meshes(RES_DIR / "meshes_ldi.npz", mesher, isovalues, mesh_box, extrema=extrema,
                          shell_delta=SHELL_DELTA * spacing, shell_half=SHELL_HALF,
                          shell_tries=4 if extrema else 0)
m = np.load(RES_DIR / "meshes_ldi.npz")
print({h: len(m[f"sheet{h}_f"]) for h in range(len(isovalues))}, "faces per surface;", len(extrema), "extrema")
# the figure's section, widened like the meshes (07f draws it)
inp = np.load(RES_DIR / "inputs.npz")
sec_x, sec_y, sec_pts = isocline_inputs.figure_section(inp["x"], inp["y"], inp["z"], data["horizons"], MESH_PAD_X)
with torch.no_grad():
    sec_phi = np.asarray(model.predict(torch.as_tensor(sec_pts, dtype=curlew.dtype), litho=False).scalar, dtype=float)

def section_gradient(p):
    # the field's exact gradient (curlew), in the plane of the section: its x and z components
    q = torch.as_tensor(np.c_[p[:, 0], np.full(len(p), sec_y), p[:, 1]], dtype=curlew.dtype)
    g, _ = event.field.gradient(q, normalize=False, return_value=True)
    return g.detach().cpu().numpy()[:, [0, 2]]


# its interior extrema: zeros of the in-plane gradient (07f draws them)
sec_crit = diagnostics.find_critical_points(None, sec_x, inp["z"], gradient=section_gradient)
print(f"section critical points: {sec_crit.counts}")
np.savez_compressed(RES_DIR / "section_ldi.npz", section_x=sec_x, section_y=sec_y,
                    section_phi=sec_phi.reshape(len(sec_x), len(inp["z"])).astype(np.float32),
                    **diagnostics.pack_critical(sec_crit, prefix="section_critical_"))
print(f"wrote section_ldi.npz: y = {sec_y:.0f} m, x from {sec_x[0]:.0f} to {sec_x[-1]:.0f} m")


{0: 175514, 1: 187706, 2: 160782} faces per surface; 0 extrema
section critical points: {'maximum': 0, 'minimum': 0, 'saddle': 0, 'degenerate': 0}
wrote section_ldi.npz: y = -30 m, x from -360 to 290 m


## Truth surfaces

Each truth polyline (in the profile, `(u, w)`) swept along the hinge from one end of the box
to the other, `N_ALONG` rows of vertices long, as a strip of triangles.

In [6]:
N_ALONG = 24
u_ax, w_ax, hinge = data["u_ax"], data["w_ax"], data["hinge"]
along = np.linspace(box[0, 1], box[1, 1], N_ALONG)       # hinge is along y (07a)


def sweep(h, bounds):
    verts, faces, start = [], [], 0
    for line in data["truth_lines"][int(h)]:
        n = len(line)
        rows = [line[:, :1] * u_ax + line[:, 1:] * w_ax + t * hinge for t in along]
        verts.append(np.vstack(rows))
        i = np.arange(n - 1)[None, :] + n * np.arange(N_ALONG - 1)[:, None] + start
        quads = np.stack([i, i + 1, i + n + 1, i + n], axis=-1).reshape(-1, 4)
        faces.append(np.r_[quads[:, [0, 1, 2]], quads[:, [0, 2, 3]]])
        start += n * N_ALONG
    return np.vstack(verts), np.vstack(faces)


diagnostics.export_meshes(RES_DIR / "meshes_truth.npz", sweep, np.arange(len(data["horizon_names"])), box,
                          shell_delta=0.0, shell_half=0.0, shell_tries=0)
m = np.load(RES_DIR / "meshes_truth.npz")
print({h: len(m[f"sheet{h}_f"]) for h in range(len(data["horizon_names"]))}, "faces per truth surface")

{0: 18354, 1: 18354, 2: 18354} faces per truth surface
